<a href="https://colab.research.google.com/github/ludanajoshi/BIS-Lab/blob/main/BIS_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

# ==========================================
# 1. PROBLEM SETUP & CONSTRAINTS
# ==========================================
# Available schedule slots (3 days, 2 sessions per day = 6 slots)
SLOTS = [
    "Day 1 - Morning", "Day 1 - Afternoon",
    "Day 2 - Morning", "Day 2 - Afternoon",
    "Day 3 - Morning", "Day 3 - Afternoon"
]

EXAMS = ["Math", "Physics", "Chemistry", "Biology", "History", "English"]

# Which student groups are enrolled in which exams
STUDENT_GROUPS = {
    "Group_A": ["Math", "Physics", "Chemistry"],
    "Group_B": ["Math", "Biology", "English"],
    "Group_C": ["Physics", "History"],
    "Group_D": ["Chemistry", "Biology", "History"]
}

# GA Hyperparameters
POPULATION_SIZE = 50
GENERATIONS = 200
MUTATION_RATE = 0.15

# ==========================================
# 2. CORE GENETIC ALGORITHM MECHANICS
# ==========================================

def create_chromosome():
    """
    Creates a random timetable solution.
    Chromosome structure: A dictionary mapping {Exam: Slot_Index}
    """
    chromosome = {}
    for exam in EXAMS:
        chromosome[exam] = random.randint(0, len(SLOTS) - 1)
    return chromosome

def calculate_fitness(chromosome):
    """
    The Fitness Function (Scoring System).
    Starts at 100 points. Loses points for every student conflict (clash).
    """
    clashes = 0

    # Check each student group to see if their exams overlap in the same slot
    for group, enrolled_exams in STUDENT_GROUPS.items():
        # Find the scheduled slots for this group's exams
        scheduled_slots = [chromosome[exam] for exam in enrolled_exams if exam in chromosome]

        # If the number of unique slots is less than the total exams, there is a clash
        unique_slots = set(scheduled_slots)
        clashes += (len(scheduled_slots) - len(unique_slots))

    # Fitness formula: perfect score is 100. Every clash penalizes by 20 points.
    fitness_score = 100 - (clashes * 20)
    return max(0, fitness_score)

def selection(population):
    """
    Selects a parent using Tournament Selection.
    """
    pool = random.sample(population, 3)
    pool.sort(key=lambda x: calculate_fitness(x), reverse=True)
    return pool[0]

def crossover(parent1, parent2):
    """
    Combines two parent schedules using a Single-Point Crossover.
    """
    child = {}
    crossover_point = random.randint(1, len(EXAMS) - 1)

    for i, exam in enumerate(EXAMS):
        if i < crossover_point:
            child[exam] = parent1[exam]
        else:
            child[exam] = parent2[exam]
    return child

def mutate(chromosome):
    """
    Randomly alters a few exam slots based on the mutation rate.
    """
    for exam in EXAMS:
        if random.random() < MUTATION_RATE:
            chromosome[exam] = random.randint(0, len(SLOTS) - 1)
    return chromosome

# ==========================================
# 3. THE EVOLUTIONARY RUN LOOP
# ==========================================

def run_genetic_algorithm():
    # Step 1: Initialize random population
    population = [create_chromosome() for _ in range(POPULATION_SIZE)]

    print("Starting Evolution Loop...\n")

    for generation in range(GENERATIONS):
        # Sort population by fitness scores
        population.sort(key=lambda x: calculate_fitness(x), reverse=True)
        best_current = population[0]
        best_fitness = calculate_fitness(best_current)

        # Early exit condition: standard collision-free schedule found
        if best_fitness == 100:
            print(f"🎉 Perfect clash-free schedule evolved at Generation {generation}!")
            return best_current

        # Breed a new generation
        new_generation = [best_current] # Carry over the best (Elitism)

        while len(new_generation) < POPULATION_SIZE:
            p1 = selection(population)
            p2 = selection(population)
            child = crossover(p1, p2)
            child = mutate(child)
            new_generation.append(child)

        population = new_generation

    print("⚠️ Reached max generations without perfect score.")
    return population[0]

# ==========================================
# 4. EXECUTE AND PRINT THE TIMETABLE
# ==========================================
if __name__ == "__main__":
    best_timetable = run_genetic_algorithm()

    print("\n==============================")
    print("      EVOLVED EXAM TIMETABLE  ")
    print("==============================")

    # Group exams by their time slots for clean printing
    slots_display = {i: [] for i in range(len(SLOTS))}
    for exam, slot_idx in best_timetable.items():
        slots_display[slot_idx].append(exam)

    for slot_idx, assigned_exams in slots_display.items():
        exams_str = ", ".join(assigned_exams) if assigned_exams else "No Exams"
        print(f"📍 {SLOTS[slot_idx]:<20} : {exams_str}")

    print(f"\nFinal Schedule Fitness Score: {calculate_fitness(best_timetable)}/100")


Starting Evolution Loop...

🎉 Perfect clash-free schedule evolved at Generation 0!

      EVOLVED EXAM TIMETABLE  
📍 Day 1 - Morning      : History
📍 Day 1 - Afternoon    : Physics, Biology
📍 Day 2 - Morning      : No Exams
📍 Day 2 - Afternoon    : Math
📍 Day 3 - Morning      : English
📍 Day 3 - Afternoon    : Chemistry

Final Schedule Fitness Score: 100/100
